In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEndpoint
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema 
import os

os.environ["HF_HOME"] = "/Users/aryan/Downloads/hf_cache"

In [6]:
# llm = HuggingFacePipeline.from_model_id(
#     model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     task = "text-generation",
#     pipeline_kwargs={
#         "temperature":0.3
#     }
# )

# # llm = HuggingFaceEndpoint(
# #     repo_id="google/gemma-2-2b-it",
# #     task = "text-generation"
# # )

 
# model = ChatHuggingFace(llm=llm)


model = ChatGroq(
    model="qwen/qwen3.8-27b"
)

In [7]:
schema = [
    ResponseSchema(name="fact1", description="Fact 1 on the topic"),
    ResponseSchema(name="fact2", description="Fact 2 on the topic"),
    ResponseSchema(name="fact3", description="Fact 3 on the topic")
]

parser = StructuredOutputParser.from_response_schemas(schema)

template = PromptTemplate(
    template = "give 3 intrestign funfacts about the topic: {topic}\n{format_instructions}",
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [8]:
prompt = template.invoke({"topic":"black hole"})

response = model.invoke(prompt)
final_response = parser.parse(response.content)
final_response

{'fact1': 'Black holes can be so massive that if our Sun were replaced by a black hole of the same mass, Earth would continue orbiting normally, but it would be pitch dark and freezing cold because no light or heat would be emitted.',
 'fact2': 'According to Hawking radiation, black holes are not completely black; they emit faint thermal radiation due to quantum effects near the event horizon, which causes them to slowly lose mass and eventually evaporate over extremely long periods.',
 'fact3': 'The supermassive black hole at the center of our Milky Way galaxy, known as Sagittarius A*, has a mass equivalent to about 4 million suns, yet it fits within a region smaller than our solar system.'}

In [9]:
chain = template | model | parser
response = chain.invoke({"topic":"black hole"})

response

{'fact1': 'Time dilation near a black hole is so extreme that if you were to hover just above the event horizon, one hour for you could equate to seven years for someone far away, a phenomenon famously depicted in the movie Interstellar.',
 'fact2': "Black holes are not actually black; if you could somehow observe them from outside while ignoring the gravitational lensing, the surroundings would appear distorted and magnified, creating a 'lens' effect that makes the background stars look like a ring around the hole.",
 'fact3': 'Supermassive black holes at the centers of galaxies can be billions of times more massive than our Sun, yet they can be smaller in diameter than our solar system, meaning their density is so high that a teaspoon of their material would weigh more than Mount Everest.'}